# nonSNP-specific climate-hit genes (clq0.9, all 20 axes)

Genes on clq0.9 blocks that are WZA **BH q<0.05 in non-SNP (SV+small-indel) but NOT
in SNP**, for at least one of the 20 climate axes (bio1..bio19 + PC1) x 3 models
(kendall / lfmm-no-gif / quasi-binomial). These are regions where structural /
indel variation carries a climate association that SNPs miss.

Source: `multiaxis/nonsnp_specific_genes.csv` (block -> clq0.9 span -> overlapping
TAIR10 genes -> Ensembl Plants symbol + description). Provenance columns:
- `n_flags` = # of (axis,model) combos flagging the block nonSNP-specific
- `axes_nonsnp_specific`, `models` = which axes / models
- `never_snp_hit` = block is NEVER a SNP BH-hit on any axis/model (strongest nonSNP-only evidence)

**Caveat:** the pooled set includes the (uncalibrated, inflated) Kendall model; the
`models` column lets you restrict to the calibrated models (lfmm / binomial). The
`never_snp_hit=True` + high-`n_flags` rows are robust regardless.

> ## ⚠️ SUPERSEDED (2026-07-27) — do not cite this notebook's "heat-stress dominates" read
>
> This notebook was built on deg-2 WZA + the retired pooled 2-class (`snp`/`nonsnp`) split,
> both since replaced (isotonic SD correction + `snp`/`sv`/`smallindel` 3-class split — see
> `../STATUS_clq90.md` §6). Re-running the identical annotation pass on the corrected output
> (`../multiaxis/candidate_genes_corrected.py`) gives a materially different answer:
> **336 genes total, only 8 (2.4%) in any stress category** (5 heat / 2 drought-ABA / 1 cold /
> 0 flowering-circadian) — background rate, not a theme. Only HSBP/AT4G15802 survives from the
> 11-gene heat list below; HSFA2, the HSP20 cluster, RAS1, COR15A, and VRN2 do not appear in the
> corrected candidate set. **Conclusion: no functional theme survives correction.** See
> `../STATUS_clq90.md` §6 and `../multiaxis/candidate_genes_corrected.csv` for the current result.
> The cells below are kept for provenance only.

In [1]:
import pandas as pd, numpy as np
pd.set_option("display.max_rows", 600)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

CSV = "/global/scratch/users/tbellg/kmate/analysis/grenenet_selection/r2_gea_nonsnp/phase1_replication/results/multiaxis/nonsnp_specific_genes.csv"
d = pd.read_csv(CSV)
# one row per gene (a gene can appear via >1 block; keep the block with most flags)
g = (d[d.gene != ""].sort_values("n_flags", ascending=False)
     .drop_duplicates("gene").reset_index(drop=True))
g["description"] = g["description"].fillna(""); g["symbol"] = g["symbol"].fillna("")
print(f"{d.block.nunique()} nonSNP-specific blocks | {len(g)} unique genes "
      f"({(g.symbol!='').sum()} with symbols) | "
      f"{d[d.never_snp_hit==True].block.nunique()} blocks never a SNP hit")
g[["block","symbol","gene","description","n_flags","models","never_snp_hit"]].head(15)

437 nonSNP-specific blocks | 548 unique genes (540 with symbols) | 305 blocks never a SNP hit


,block,symbol,gene,description,n_flags,models,never_snp_hit
0,Chr1_6372,,NaN,,19,binomial;kendall;lfmm,False
1,Chr5_6500,AT5G38100,AT5G38100,S-adenosyl-L-methionine-dependent methyltransferases superfamily protein,17,binomial;kendall;lfmm,True
2,Chr1_3032,AT1G17300,AT1G17300,uncharacterized protein,16,binomial;kendall,True
3,Chr5_4219,SIGE,AT5G24120,sigma factor E,16,binomial;kendall;lfmm,True
4,Chr3_3435,AT3G20280,AT3G20280,RING/FYVE/PHD zinc finger superfamily protein,14,kendall,True
5,Chr4_6488,CSLG3,AT4G23990,cellulose synthase like G3,14,binomial;kendall,False
6,Chr1_14721,AT1G74910,AT1G74910,ADP-glucose pyrophosphorylase family protein,13,kendall,True
7,Chr1_3030,AlaAT1,AT1G17290,alanine aminotransferas,12,binomial;kendall,True
8,Chr4_2380,AT4G10540,AT4G10540,Subtilase family protein,11,lfmm,True
9,Chr4_5141,BEH3,AT4G18890,BES1/BZR1 homolog 3,11,kendall,True


## Functional-category tagging

Tag each gene by keyword in its symbol+description, plus a curated exact-symbol
list for well-known Arabidopsis genes whose description is terse.

In [2]:
CATS = {
 "heat":       r"heat|thermo|high temperature|chaperone|heat shock|hsp|hsf|dnaj",
 "drought_ABA":r"drought|abscisic|\baba\b|dehydrat|desicc|osmotic|water stress|stomat|dehydrin|late embryogenesis|proline|salt",
 "cold":       r"\bcold\b|freezing|chilling|\bcbf\b|frost|cor15",
 "flower_circ":r"flower|floral|photoperiod|vernaliz|infloresc|florigen|circadian|clock|rhythm|oscillat",
}
CURATED = {  # exact symbol -> category (catches terse descriptions)
 "HSFA2":"heat","HSBP":"heat","HSP17.4":"heat","MBF1C":"heat",
 "RAS1":"drought_ABA","DREB2A":"drought_ABA","RD29A":"drought_ABA","NCED3":"drought_ABA",
 "COR15A":"cold","VRN2":"flower_circ","FT":"flower_circ","FLC":"flower_circ",
 "GI":"flower_circ","CCA1":"flower_circ","TOC1":"flower_circ","PIF4":"flower_circ",
}
txt = (g.symbol + " " + g.description).str.lower()
g["category"] = ""
for cat, pat in CATS.items():
    g.loc[(g.category == "") & txt.str.contains(pat, regex=True, na=False), "category"] = cat
for sym, cat in CURATED.items():
    g.loc[g.symbol == sym, "category"] = cat
print("category counts:\n", g["category"].replace("", "other").value_counts().to_string())

category counts:
 category
other          530
heat            11
drought_ABA      5
flower_circ      1
cold             1


## Candidate genes by functional category (stress / phenology)

The scientifically interesting subset. HEAT dominates; flowering/circadian is nearly absent.

In [3]:
for cat in ["heat", "drought_ABA", "cold", "flower_circ"]:
    sub = g[g.category == cat].sort_values("n_flags", ascending=False)
    print(f"\n===== {cat}: {len(sub)} genes =====")
    if len(sub):
        print(sub[["block","symbol","gene","description","n_flags","axes_nonsnp_specific","never_snp_hit"]]
              .to_string(index=False))


===== heat: 11 genes =====
     block    symbol      gene                                                              description  n_flags                 axes_nonsnp_specific  never_snp_hit
 Chr4_4191      HSBP AT4G15802                                        heat shock factor binding protein       10 bio1;bio16;bio18;bio19;bio3;bio5;pc1          False
 Chr1_9404 AT1G54850 AT1G54850                                HSP20-like chaperones superfamily protein        5                    bio14;bio17;bio18           True
 Chr1_9404 AT1G54840 AT1G54840                                HSP20-like chaperones superfamily protein        5                    bio14;bio17;bio18           True
 Chr5_2542 AT5G16210 AT5G16210                                           HEAT repeat-containing protein        4                 bio19;bio2;bio7;bio8           True
 Chr3_1035 AT3G06340 AT3G06340                     DNAJ heat shock N-terminal domain-containing protein        2                           bio12;bi

## Full nonSNP-specific gene list (with descriptions)

All unique genes, sorted by how many axis-model combos flag them. Written also to
`multiaxis/nonsnp_specific_genes_table.csv` (one row per gene).

In [4]:
full = g[["block","chrom","start","end","symbol","gene","description",
                          "category","n_flags","axes_nonsnp_specific","models","never_snp_hit"]].copy()
full.to_csv("/global/scratch/users/tbellg/kmate/analysis/grenenet_selection/r2_gea_nonsnp/phase1_replication/results/"
            "multiaxis/nonsnp_specific_genes_table.csv", index=False)
full

,block,chrom,start,end,symbol,gene,description,category,n_flags,axes_nonsnp_specific,models,never_snp_hit
0,Chr1_6372,Chr1,11710262,11710579,,NaN,,,19,bio10;bio12;bio14;bio15;bio16;bio17;bio18;bio19;bio3;bio6;pc1,binomial;kendall;lfmm,False
1,Chr5_6500,Chr5,15194321,15208171,AT5G38100,AT5G38100,S-adenosyl-L-methionine-dependent methyltransferases superfamily protein,,17,bio1;bio11;bio13;bio15;bio16;bio17;bio18;bio3;bio4;bio5;bio6;bio7;bio9,binomial;kendall;lfmm,True
2,Chr1_3032,Chr1,5927494,5927643,AT1G17300,AT1G17300,uncharacterized protein,,16,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio19;bio3;bio5;bio9;pc1,binomial;kendall,True
3,Chr5_4219,Chr5,8155850,8161292,SIGE,AT5G24120,sigma factor E,,16,bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio5;bio6;bio9;pc1,binomial;kendall;lfmm,True
4,Chr3_3435,Chr3,7071542,7072177,AT3G20280,AT3G20280,RING/FYVE/PHD zinc finger superfamily protein,,14,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio19;bio4;bio6;bio7;bio9;pc1,kendall,True
5,Chr4_6488,Chr4,12459192,12460974,CSLG3,AT4G23990,cellulose synthase like G3,,14,bio12;bio14;bio15;bio17;bio18;bio19;bio4;bio5;bio6;bio7;pc1,binomial;kendall,False
6,Chr1_14721,Chr1,28136604,28138650,AT1G74910,AT1G74910,ADP-glucose pyrophosphorylase family protein,,13,bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio19;bio4;bio6;bio7;bio9;pc1,kendall,True
7,Chr1_3030,Chr1,5925513,5926266,AlaAT1,AT1G17290,alanine aminotransferas,,12,bio1;bio10;bio11;bio12;bio14;bio17;bio19;bio3;bio5;bio9;pc1,binomial;kendall,True
8,Chr4_2380,Chr4,6514184,6514620,AT4G10540,AT4G10540,Subtilase family protein,,11,bio1;bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio5;bio9;pc1,lfmm,True
9,Chr4_5141,Chr4,10352996,10353157,BEH3,AT4G18890,BES1/BZR1 homolog 3,,11,bio10;bio11;bio12;bio14;bio15;bio17;bio18;bio19;bio6;bio7;pc1,kendall,True
